# Amazon Price Tracker with Scavio API

Track and compare Amazon product prices in real time using the Scavio search API and LangChain. A free alternative to Keepa and CamelCamelCamel for competitor price monitoring.

**What you will learn:**
- Search Amazon products by keyword with ScavioAmazonSearch
- Pull detailed product data (price, rating, reviews) with ScavioAmazonProduct
- Build a price comparison table across products
- Monitor competitor pricing in any category

**Prerequisites:**
- Free Scavio API key (50 free credits (one-time)): https://dashboard.scavio.dev
- OpenAI API key

**Tools used:** ScavioAmazonSearch, ScavioAmazonProduct

In [1]:
# pip install langchain langchain-openai langchain-scavio python-dotenv

In [2]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_scavio import ScavioAmazonSearch, ScavioAmazonProduct

load_dotenv(override=True)

True

In [3]:
SYSTEM_PROMPT = """You are AmazonPriceTracker, a competitor price monitoring agent.

Workflow:
1. Take the user's product category or keyword.
2. Call ScavioAmazonSearch to find the top 5 products in that category.
3. Call ScavioAmazonProduct on each of the top 3 ASINs to get full
   pricing details (price, list price, availability, ratings).
4. Compile a price monitoring report:

   ## Price Report: <category>

   ### Price Comparison Table
   | Product | ASIN | Price | Rating | Reviews | Availability |
   |---------|------|-------|--------|---------|-------------|
   (fill from tool output)

   ### Price Analysis
   - Cheapest option: <product + price>
   - Best value (price-to-rating ratio): <product>
   - Premium option: <product + price + why>

   ### Competitive Insights
   - Price range in this category: $X - $Y
   - Average rating: X.X stars
   - Key differentiators between top and bottom priced options

Rules:
- Never invent ASINs, prices, or ratings. Only use tool output.
- Call only ONE tool per step.
- Keep the final report under 300 words.
"""

In [4]:
def build_agent():
    model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
    tools = [
        ScavioAmazonSearch(max_results=5),
        ScavioAmazonProduct(),
    ]
    return create_agent(model, tools=tools, system_prompt=SYSTEM_PROMPT)

In [5]:
agent = build_agent()
result = agent.invoke({
    "messages": [{"role": "user", "content": "mechanical keyboards under $100"}]
})
print(result["messages"][-1].content)

## Price Report: Mechanical Keyboards Under $100

### Price Comparison Table
| Product | ASIN | Price | Rating | Reviews | Availability |
|---------|------|-------|--------|---------|--------------|
| RisoPhy Mechanical Gaming Keyboard, RGB 104 Keys Ultra-Slim LED Backlit USB Wired Keyboard with Blue Switch | B09TR4Y91J | $28.99 | 4.5 | 3,226 | In Stock, Free delivery by June 3 |
| Redragon Mechanical Gaming Keyboard Wired, 11 Programmable Backlit Modes, Hot-Swappable Red Switch | B0CF3VGQFL | $29.99 | 4.3 | 6,549 | In Stock, Free delivery by June 3 |
| Redragon K668 RGB Gaming Keyboard, 108 Keys Wired Mechanical Keyboard w/Extra 4 Hotkeys | B0CDWP1D58 | $39.99 | 4.5 | 2,000 | In Stock, Free delivery by June 3 |

### Price Analysis
- Cheapest option: RisoPhy Mechanical Gaming Keyboard at $28.99
- Best value (price-to-rating ratio): RisoPhy Mechanical Gaming Keyboard with 4.5 stars and $28.99 price
- Premium option: Redragon K668 RGB Gaming Keyboard at $39.99, offers extra hotkeys and R

## Next Steps

- Monitor any product category or specific competitor ASINs
- Run weekly to track price changes over time
- Compare pricing across seasons or sales events
- Combine with ScavioWalmartSearch for cross-platform price monitoring

**Credits used:** ~4-5 per run (one search + three product lookups)